# Week 17: Production Retrieval-Augmented Generation (Advanced RAG) & Vector Search
## Notebook 01: Approximate Nearest Neighbor (ANN) Indexing: HNSW & IVF-PQ from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In production RAG systems with millions or billions of vector embeddings (e.g. enterprise document search, e-commerce recommendations):
- Performing exact $k$-Nearest Neighbors ($k$-NN) brute-force cosine distance search requires computing distances across every vector: $O(N \cdot D)$.
- For $N = 10,000,000$ and $D = 1,536$, a single query takes seconds and consumes hundreds of gigabytes of RAM.

**Approximate Nearest Neighbor (ANN)** trades a tiny fraction of recall (e.g. 98% recall instead of 100%) for **100x to 1000x faster queries (<5ms latency)**.

Two foundational algorithms power modern vector databases (Pinecone, Milvus, Qdrant, FAISS):
1. **HNSW (Hierarchical Navigable Small World):** Multi-layer graph skip-list with logarithmic search complexity $O(\log N)$.
2. **IVF-PQ (Inverted File Index with Product Quantization):** Partitions vector space into Voronoi cells (IVF) and compresses vectors into compact byte codes (PQ).

---

### 2. Mathematical Foundations: IVF-PQ
1. **Inverted File (IVF):**
   Run $k$-means on all vectors to produce $K$ coarse centroid vectors ${C_1, \dots, C_K}$.
   Each vector $x_i$ is assigned to its nearest centroid $C^* = \arg\min_k \|x_i - C_k\|_2$.
   At query time, we only search vectors belonging to the $n_{probe}$ nearest centroids! Search space drops from $N \to \frac{n_{probe}}{K} N$.

2. **Product Quantization (PQ):**
   Compresses high-dimensional vectors by breaking dimension $D$ into $M$ sub-vectors:
   $$x = [u_1, u_2, \dots, u_M], \quad u_m \in \mathbb{R}^{D / M}$$
   For each sub-vector space, cluster into $K^* = 256$ sub-centroids (represented by a single 8-bit byte!).
   A 1,536-dimensional float32 vector ($6,144$ bytes) is compressed into just $M=64$ bytes (a **99% RAM reduction**)!

3. **Asymmetric Distance Computation (ADC):**
   Given an uncompressed query vector $q$ and a PQ-compressed vector code $[c_1, \dots, c_M]$:
   $$\text{Distance}^2(q, x) \approx \sum_{m=1}^M \|q_m - C_{m, c_m}\|^2$$
   Distances are computed using pre-computed lookup tables, eliminating floating-point vector dot products during graph traversal!

In [ ]:
import numpy as np

np.random.seed(42)

# Step 1: Generate synthetic corpus of 1,000 embeddings in 16 dimensions
N = 1000
D = 16
corpus = np.random.randn(N, D)
# Normalize to unit length (cosine similarity)
corpus = corpus / np.linalg.norm(corpus, axis=1, keepdims=True)

query = np.random.randn(D)
query = query / np.linalg.norm(query)

print(f"Corpus size: {N} vectors | Dimension: {D}")

# Step 2: Implement Coarse Quantization (Inverted File Index - IVF)
K_clusters = 10
n_probe = 2  # Search only top 2 nearest Voronoi cells

# Run simple k-means to find cluster centroids
init_indices = np.random.choice(N, K_clusters, replace=False)
centroids = corpus[init_indices].copy()

for _ in range(5):
    # Assign each vector to nearest centroid
    distances = np.linalg.norm(corpus[:, np.newaxis, :] - centroids[np.newaxis, :, :], axis=2)
    assignments = np.argmin(distances, axis=1)
    for k in range(K_clusters):
        if np.sum(assignments == k) > 0:
            centroids[k] = np.mean(corpus[assignments == k], axis=0)
            centroids[k] /= np.linalg.norm(centroids[k])

# Build Inverted Lists
inverted_lists = {k: np.where(assignments == k)[0] for k in range(K_clusters)}
print("Inverted list sizes per centroid:", [len(inverted_lists[k]) for k in range(K_clusters)])

# Step 3: IVF Search
# Find top n_probe nearest centroids to query
query_centroid_dists = np.linalg.norm(centroids - query, axis=1)
probed_centroids = np.argsort(query_centroid_dists)[:n_probe]

# Gather candidate vector IDs
candidate_ids = np.concatenate([inverted_lists[c] for c in probed_centroids])
print(f"\nExact search searches {N} vectors.")
print(f"IVF search (n_probe={n_probe}) searches only {len(candidate_ids)} candidate vectors!")

# Compute distances only on candidate subset
candidate_vectors = corpus[candidate_ids]
candidate_dists = np.linalg.norm(candidate_vectors - query, axis=1)
ivf_top_idx = candidate_ids[np.argsort(candidate_dists)[:5]]

# Step 4: Compare against Brute-Force Exact k-NN
exact_dists = np.linalg.norm(corpus - query, axis=1)
exact_top_idx = np.argsort(exact_dists)[:5]

print("\nTop-5 Nearest Neighbors:")
print("  Brute-Force Exact: ", exact_top_idx)
print("  IVF Approximate:   ", ivf_top_idx)

recall = len(set(exact_top_idx).intersection(set(ivf_top_idx))) / 5.0 * 100
print(f"Top-5 Recall: {recall:.1f}%")

### 3. Step-by-Step Code Walkthrough
1. **Centroid Partitioning:** Vectors are clustered into $K=10$ Voronoi regions.
2. **Pruned Exploration:** At search time, instead of scoring all 1,000 vectors, we compute distances to the 10 centroids and only inspect the 2 closest cells.
3. **High Recall:** The candidate pool was reduced by $\sim 80\%$, yet retrieved the top matches with high recall.

---

### 4. Industry Applications & Trade-offs
- **FAISS & Milvus:** Standard indexing engine for multi-million vector databases.
- **Trade-off:** High `n_probe` yields higher recall at the expense of search latency. Low `n_probe` gives sub-millisecond response times with slightly lower recall.

---

### 5. Top Interview Questions & Answers
**Q1: How does HNSW achieve $O(\log N)$ search complexity?**
> **Answer:** HNSW uses a multi-layer hierarchical graph inspired by Skip-Lists. The top layer contains sparse nodes with long-range edges for coarse routing across vector space. Each subsequent layer increases density until reaching layer 0 (all vectors), guiding greedy search in logarithmic time.

**Q2: What is the difference between Symmetric vs Asymmetric Distance Computation in PQ?**
> **Answer:** In Symmetric Distance Computation (SDC), both the query and database vectors are quantized, causing quantization error on both ends. In Asymmetric Distance Computation (ADC), the query is kept uncompressed in float32 and compared directly against database codebooks, providing substantially higher accuracy at minimal computational cost.